# 8 Igualamos tama?o

Flujo para abrir H&E + HSI, segmentar la H&E, detectar la caja azul ajustada en HSI y asignarle las medidas reales.

In [ ]:
from pathlib import Path
from contextlib import redirect_stdout
import io

import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
import numpy as np
import openslide
from PIL import Image
from scipy import ndimage as ndi

plt.rcParams['figure.dpi'] = 120


In [ ]:
SPECIMENS = {
    'SB012': {
        'he_path': Path(r'Datos\SB012\SB012\H&E\SB012\SB012.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm_edu.hdr'),
    },
    'SB013': {
        'he_path': Path(r'Datos\SB013\SB013\H&E\SB013.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB013\SB013\SB013_001\SB013_nrm_edu.hdr'),
    },
    'SB017': {
        'he_path': Path(r'Datos\SB017_uomo\SB017_uomo\H&E\SB017\SB017.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_nrm_edu.hdr'),
    },
    'SB018': {
        'he_path': Path(r'Datos\SB018\SB018\H&E\SB018\SB018.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB018\SB018\SB018_001\SB018_nrm_edu.hdr'),
    },
    'SB019': {
        'he_path': Path(r'Datos\SB019\SB019\H&E\SB019\SB019.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB019\SB019\SB019_001\SB019_nrm_edu.hdr'),
    },
    'SB020': {
        'he_path': Path(r'Datos\SB020\SB020\H&E\SB020\SB020.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB020\SB020\SB020_001\SB020_nrm_edu.hdr'),
    },
}

def check_specimen_paths(specimens=SPECIMENS):
    for specimen_id, paths in specimens.items():
        print(specimen_id)
        print('  H&E:', paths['he_path'].exists(), '|', paths['he_path'])
        print('  HSI:', paths['hsi_hdr_path'].exists(), '|', paths['hsi_hdr_path'])



## Funciones reutilizadas de `7_Deteccion_caja_azul_HSI.ipynb`

Se incluyen las funciones de lectura ENVI, detección de caja azul y reducción del ROI ajustado.

In [ ]:
def parse_envi_header(hdr_path):
    hdr_path = Path(hdr_path)
    text = hdr_path.read_text(encoding='utf-8', errors='ignore')
    metadata = {}
    current_key = None
    current_value = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.upper() == 'ENVI':
            continue

        if current_key is not None:
            current_value.append(line)
            if '}' in line:
                metadata[current_key] = ' '.join(current_value)
                current_key = None
                current_value = []
            continue

        if '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()

        if value.startswith('{') and '}' not in value:
            current_key = key
            current_value = [value]
        else:
            metadata[key] = value

    return metadata


def parse_wavelengths(metadata):
    values = metadata.get('wavelength')
    if values is None:
        return None

    values = values.strip().strip('{}')
    return np.array([float(v.strip()) for v in values.split(',') if v.strip()], dtype=np.float32)


def envi_dtype(metadata):
    data_type = int(metadata['data type'])
    byte_order = int(metadata.get('byte order', 0))
    endian = '<' if byte_order == 0 else '>'
    dtype_map = {
        1: 'u1',
        2: 'i2',
        3: 'i4',
        4: 'f4',
        5: 'f8',
        12: 'u2',
        13: 'u4',
        14: 'i8',
        15: 'u8',
    }
    if data_type not in dtype_map:
        raise ValueError(f'Tipo de dato ENVI no soportado: {data_type}')
    return np.dtype(endian + dtype_map[data_type])


def find_envi_data_path(hdr_path, metadata):
    hdr_path = Path(hdr_path)
    candidates = []

    for key in ('data file', 'file name', 'image'):
        if key in metadata:
            value = metadata[key].strip().strip('{}').strip('"')
            candidates.append(hdr_path.parent / value)

    candidates.extend([
        hdr_path.with_suffix(''),
        hdr_path.parent / hdr_path.stem.replace('_edu', ''),
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f'No se encontro el binario ENVI asociado a {hdr_path.name}')


def read_envi_band(hdr_path, metadata, band_idx):
    data_path = find_envi_data_path(hdr_path, metadata)
    samples = int(metadata['samples'])
    lines = int(metadata['lines'])
    bands = int(metadata['bands'])
    offset = int(metadata.get('header offset', 0))
    interleave = metadata.get('interleave', 'bsq').strip().lower()
    dtype = envi_dtype(metadata)

    if interleave == 'bsq':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(bands, lines, samples))
        return np.asarray(cube[band_idx], dtype=np.float32)
    if interleave == 'bil':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, bands, samples))
        return np.asarray(cube[:, band_idx, :], dtype=np.float32)
    if interleave == 'bip':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, samples, bands))
        return np.asarray(cube[:, :, band_idx], dtype=np.float32)

    raise ValueError(f'Interleave ENVI no soportado: {interleave}')


def robust_normalize(channel, p_low=2, p_high=98):
    channel = np.asarray(channel, dtype=np.float32)
    if not np.any(np.isfinite(channel)):
        return np.zeros_like(channel, dtype=np.float32)

    lo = np.nanpercentile(channel, p_low)
    hi = np.nanpercentile(channel, p_high)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(channel, dtype=np.float32)

    normalized = np.clip((channel - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(normalized, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def hsi_pseudo_rgb_from_path(hdr_path, r_nm=650, g_nm=550, b_nm=450):
    hdr_path = Path(hdr_path)
    if not hdr_path.exists():
        raise FileNotFoundError(f'No existe la HSI: {hdr_path}')

    metadata = parse_envi_header(hdr_path)
    wavelengths = parse_wavelengths(metadata)

    if wavelengths is None:
        band_count = int(metadata['bands'])
        r_idx = min(band_count - 1, int(round(0.72 * (band_count - 1))))
        g_idx = min(band_count - 1, int(round(0.50 * (band_count - 1))))
        b_idx = min(band_count - 1, int(round(0.28 * (band_count - 1))))
        band_info = {'r_idx': r_idx, 'g_idx': g_idx, 'b_idx': b_idx}
    else:
        r_idx = int(np.argmin(np.abs(wavelengths - r_nm)))
        g_idx = int(np.argmin(np.abs(wavelengths - g_nm)))
        b_idx = int(np.argmin(np.abs(wavelengths - b_nm)))
        band_info = {
            'r_idx': r_idx,
            'r_nm': float(wavelengths[r_idx]),
            'g_idx': g_idx,
            'g_nm': float(wavelengths[g_idx]),
            'b_idx': b_idx,
            'b_nm': float(wavelengths[b_idx]),
        }

    rgb = np.stack(
        [
            robust_normalize(read_envi_band(hdr_path, metadata, r_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, g_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, b_idx)),
        ],
        axis=-1,
    )
    rgb = np.nan_to_num(rgb, nan=0.0, posinf=1.0, neginf=0.0)
    rgb = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

    info = {
        'shape': rgb.shape,
        'data_path': find_envi_data_path(hdr_path, metadata),
        **band_info,
    }
    return rgb, info



In [ ]:
def draw_rectangle_rgb(img_rgb, bbox, color=(255, 255, 0), thickness=4):
    out = img_rgb.copy()
    x1, y1, x2, y2 = bbox
    h, w = out.shape[:2]
    x1 = int(np.clip(x1, 0, w - 1))
    x2 = int(np.clip(x2, 0, w - 1))
    y1 = int(np.clip(y1, 0, h - 1))
    y2 = int(np.clip(y2, 0, h - 1))
    t = max(1, int(thickness))

    out[y1:y1 + t, x1:x2 + 1] = color
    out[max(y2 - t + 1, 0):y2 + 1, x1:x2 + 1] = color
    out[y1:y2 + 1, x1:x1 + t] = color
    out[y1:y2 + 1, max(x2 - t + 1, 0):x2 + 1] = color
    return out


def overlay_mask_rgb(img_rgb, mask, color=(0, 255, 255), alpha=0.35):
    out = img_rgb.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    mask = mask.astype(bool)
    out[mask] = (1 - alpha) * out[mask] + alpha * color_arr
    return np.clip(out, 0, 255).astype(np.uint8)


def detect_blue_box_from_rgb(
    rgb_u8,
    hue_min=0.48,
    hue_max=0.66,
    sat_min=0.10,
    value_min=0.10,
    value_max=0.90,
    blue_red_min=0.090,
    green_red_min=-0.005,
    blue_excess_min=0.075,
    white_sat_max=0.080,
    white_value_min=0.82,
    open_size=3,
    close_size=25,
    dilate_size=7,
    min_area=1200,
    padding=20,
):
    """Detecta la caja azul de la pseudo-RGB y devuelve mascara, bbox y overlay."""
    rgb_u8 = np.asarray(rgb_u8, dtype=np.uint8)
    rgb = rgb_u8.astype(np.float32) / 255.0
    hsv = rgb_to_hsv(rgb)

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    blue_hue = (h >= hue_min) & (h <= hue_max)
    blue_red = b - r
    green_red = g - r
    blue_excess = 0.65 * blue_red + 0.35 * green_red
    blue_dominance = (blue_red > blue_red_min) & (green_red > green_red_min) & (blue_excess > blue_excess_min)
    not_extreme = (v >= value_min) & (v <= value_max)
    not_white = ~((v > white_value_min) & (s < white_sat_max))
    mask = blue_hue & blue_dominance & (s >= sat_min) & not_extreme & not_white

    if open_size > 1:
        mask = ndi.binary_opening(mask, structure=np.ones((open_size, open_size), dtype=bool))
    if close_size > 1:
        mask = ndi.binary_closing(mask, structure=np.ones((close_size, close_size), dtype=bool))
    if dilate_size > 1:
        mask = ndi.binary_dilation(mask, structure=np.ones((dilate_size, dilate_size), dtype=bool))

    labels, num_labels = ndi.label(mask)
    if num_labels == 0:
        raise ValueError('No se detecto ninguna region azul. Ajusta los umbrales HSV/dominancia.')

    areas = np.bincount(labels.ravel())
    areas[0] = 0
    kept_labels = np.where(areas >= min_area)[0]
    if len(kept_labels) == 0:
        kept_labels = np.array([int(np.argmax(areas))])

    clean_mask = np.isin(labels, kept_labels)
    clean_mask = ndi.binary_fill_holes(clean_mask)

    ys, xs = np.where(clean_mask)
    if len(xs) == 0:
        raise ValueError('La mascara azul quedo vacia despues de limpiar componentes.')

    img_h, img_w = clean_mask.shape
    x1 = max(int(xs.min()) - padding, 0)
    y1 = max(int(ys.min()) - padding, 0)
    x2 = min(int(xs.max()) + padding, img_w - 1)
    y2 = min(int(ys.max()) + padding, img_h - 1)
    bbox = (x1, y1, x2, y2)

    overlay = overlay_mask_rgb(rgb_u8, clean_mask, color=(0, 220, 255), alpha=0.30)
    overlay = draw_rectangle_rgb(overlay, bbox, color=(255, 255, 0), thickness=4)

    info = {
        'bbox_xyxy': bbox,
        'bbox_width': x2 - x1 + 1,
        'bbox_height': y2 - y1 + 1,
        'mask_area_px': int(clean_mask.sum()),
        'kept_components': int(len(kept_labels)),
        'sat_min': sat_min,
        'blue_red_min': blue_red_min,
        'blue_excess_min': blue_excess_min,
    }
    return clean_mask, bbox, overlay, info


def open_hsi_and_detect_blue_box(hdr_path, specimen_id=None, show=True, **detect_kwargs):
    """Abre una HSI, genera pseudo-RGB y detecta la caja azul."""
    rgb_u8, hsi_info = hsi_pseudo_rgb_from_path(hdr_path)
    mask, bbox, overlay, box_info = detect_blue_box_from_rgb(rgb_u8, **detect_kwargs)

    result = {
        'specimen_id': specimen_id,
        'hdr_path': Path(hdr_path),
        'pseudo_rgb': rgb_u8,
        'blue_mask': mask,
        'overlay': overlay,
        'bbox_xyxy': bbox,
        'hsi_info': hsi_info,
        'box_info': box_info,
    }

    if show:
        plot_blue_box_detection(result)

    return result


def plot_blue_box_detection(result):
    specimen_id = result['specimen_id'] or result['hdr_path'].stem
    hsi_info = result['hsi_info']
    box_info = result['box_info']
    band_text = f"R={hsi_info.get('r_nm', hsi_info['r_idx']):.1f}, G={hsi_info.get('g_nm', hsi_info['g_idx']):.1f}, B={hsi_info.get('b_nm', hsi_info['b_idx']):.1f}"

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].imshow(result['pseudo_rgb'])
    axes[0].set_title(f'{specimen_id} - HSI pseudo-RGB\n{band_text}')

    axes[1].imshow(result['blue_mask'], cmap='gray')
    axes[1].set_title('Mascara caja azul')

    axes[2].imshow(result['overlay'])
    axes[2].set_title(
        'Caja azul detectada\n'
        f"bbox={box_info['bbox_xyxy']} | area={box_info['mask_area_px']} px"
    )

    for ax in axes:
        ax.axis('off')

    plt.show()



In [ ]:
def shrink_blue_box_detection(
    result,
    shrink_pixels=10,
    bbox_padding=0,
    kernel_shape='ellipse',
    show=True,
):
    """
    Reduce ligeramente el contorno de la caja azul ya detectada.

    No modifica el result original. Devuelve una copia con:
        - blue_mask_shrunk
        - overlay_shrunk
        - bbox_xyxy_shrunk
        - box_info_shrunk

    Parametros
    ----------
    result : dict
        Salida de open_hsi_and_detect_blue_box(...).
    shrink_pixels : int
        Pixeles aproximados que se erosiona la mascara azul hacia dentro.
    bbox_padding : int
        Margen extra que se vuelve a sumar al bbox despues de erosionar.
        Usa 0 para el ajuste mas estricto, o 5-15 si queda demasiado cerrado.
    kernel_shape : {'ellipse', 'rect'}
        Forma del kernel de erosion. 'ellipse' suele conservar mejor esquinas suaves.
    show : bool
        Si True, muestra comparacion entre deteccion original y reducida.
    """
    rgb_u8 = np.asarray(result['pseudo_rgb'], dtype=np.uint8)
    original_mask = np.asarray(result['blue_mask']).astype(bool)
    shrink_pixels = int(max(0, shrink_pixels))
    bbox_padding = int(max(0, bbox_padding))

    if shrink_pixels == 0:
        shrunk_mask = original_mask.copy()
    else:
        try:
            import cv2

            mask_u8 = original_mask.astype(np.uint8) * 255
            k = 2 * shrink_pixels + 1
            if kernel_shape == 'rect':
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
            else:
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
            shrunk_mask = cv2.erode(mask_u8, kernel, iterations=1) > 0
        except Exception:
            structure = np.ones((2 * shrink_pixels + 1, 2 * shrink_pixels + 1), dtype=bool)
            shrunk_mask = ndi.binary_erosion(original_mask, structure=structure)

    if not np.any(shrunk_mask):
        raise ValueError('La erosion ha vaciado la mascara. Prueba con menos shrink_pixels.')

    ys, xs = np.where(shrunk_mask)
    img_h, img_w = shrunk_mask.shape
    x1 = max(int(xs.min()) - bbox_padding, 0)
    y1 = max(int(ys.min()) - bbox_padding, 0)
    x2 = min(int(xs.max()) + bbox_padding, img_w - 1)
    y2 = min(int(ys.max()) + bbox_padding, img_h - 1)
    bbox = (x1, y1, x2, y2)

    overlay = overlay_mask_rgb(rgb_u8, shrunk_mask, color=(255, 180, 0), alpha=0.35)
    overlay = draw_rectangle_rgb(overlay, bbox, color=(255, 255, 0), thickness=4)

    adjusted = result.copy()
    adjusted['blue_mask_shrunk'] = shrunk_mask
    adjusted['overlay_shrunk'] = overlay
    adjusted['bbox_xyxy_shrunk'] = bbox
    adjusted['box_info_shrunk'] = {
        'bbox_xyxy': bbox,
        'bbox_width': x2 - x1 + 1,
        'bbox_height': y2 - y1 + 1,
        'mask_area_px': int(shrunk_mask.sum()),
        'shrink_pixels': shrink_pixels,
        'bbox_padding': bbox_padding,
        'kernel_shape': kernel_shape,
    }

    if show:
        plot_shrunk_blue_box_detection(adjusted)

    return adjusted


def plot_shrunk_blue_box_detection(adjusted_result):
    specimen_id = adjusted_result.get('specimen_id') or adjusted_result['hdr_path'].stem
    original_info = adjusted_result['box_info']
    shrunk_info = adjusted_result['box_info_shrunk']

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].imshow(adjusted_result['overlay'])
    axes[0].set_title(
        'Deteccion original\n'
        f"bbox={original_info['bbox_xyxy']}"
    )

    axes[1].imshow(adjusted_result['blue_mask_shrunk'], cmap='gray')
    axes[1].set_title(
        f"Mascara reducida | shrink={shrunk_info['shrink_pixels']} px\n"
        f"area={shrunk_info['mask_area_px']} px"
    )

    axes[2].imshow(adjusted_result['overlay_shrunk'])
    axes[2].set_title(
        f'{specimen_id} - caja ajustada\n'
        f"bbox={shrunk_info['bbox_xyxy']}"
    )

    for ax in axes:
        ax.axis('off')

    plt.show()


def shrink_all_blue_box_detections(results, shrink_pixels=10, bbox_padding=0, show=True):
    """Aplica shrink_blue_box_detection a todos los results ya calculados."""
    adjusted_results = {}
    for specimen_id, result in results.items():
        print(f"\n=== {specimen_id} | shrink={shrink_pixels}px ===")
        adjusted = shrink_blue_box_detection(
            result,
            shrink_pixels=shrink_pixels,
            bbox_padding=bbox_padding,
            show=show,
        )
        adjusted_results[specimen_id] = adjusted
        print(adjusted['box_info_shrunk'])
    return adjusted_results




## Funciones reutilizadas de `3_Segmentacion_H&E.ipynb`

Se usa la misma funcion de segmentacion H&E para mantener el comportamiento consistente.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import openslide


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=8,
    val_thresh=253,
    od_thresh=0.025,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=45,
    grow_pixels=8,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False
):
    """
    Extrae el tejido de una H&E tipo .mrxs usando el contorno externo del tejido.

    Por defecto devuelve solo:
        tissue_only_rgb

    Opcionalmente puede devolver:
        mask_tissue
        info

    Parameters
    ----------
    slide_path : str or Path
        Ruta al archivo .mrxs.

    max_dim : int
        Tamaño máximo del lado largo usado para leer la imagen a baja resolución.

    sat_thresh : int
        Umbral de saturación HSV. Más bajo detecta tejido más pálido.

    val_thresh : int
        Umbral de brillo HSV. Más alto permite zonas más claras.

    od_thresh : float
        Umbral de optical density. Más bajo detecta tejido muy claro.

    min_area : int
        Área mínima de componentes a conservar.

    open_kernel_size : int
        Kernel para eliminar ruido pequeño.

    close_kernel_size : int
        Kernel para cerrar huecos y unir zonas del tejido.

    grow_pixels : int
        Dilatación final del contorno para no cortar tejido.

    use_convex_hull : bool
        Si True, usa convex hull. Es más conservador, puede meter más fondo.

    show_original : bool
        Plotea la H&E original leída.

    show_result : bool
        Plotea el resultado final.

    debug : bool
        Muestra pasos intermedios.

    return_mask : bool
        Si True, devuelve también la máscara final.

    return_info : bool
        Si True, devuelve también información auxiliar.
    """

    slide_path = Path(slide_path)

    if not slide_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {slide_path}")

    # ============================================================
    # 1. Cargar slide a baja resolución
    # ============================================================
    slide = openslide.OpenSlide(str(slide_path))
    level_dims = slide.level_dimensions

    chosen_level = None
    for i, (w, h) in enumerate(level_dims):
        if max(w, h) <= max_dim:
            chosen_level = i
            break

    if chosen_level is None:
        chosen_level = len(level_dims) - 1

    level_w, level_h = level_dims[chosen_level]

    rgb_pil = slide.read_region(
        location=(0, 0),
        level=chosen_level,
        size=(level_w, level_h)
    ).convert("RGB")

    slide.close()

    rgb_u8 = np.array(rgb_pil, dtype=np.uint8)

    # ============================================================
    # 2. Mapas HSV y optical density
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    rgb_float = rgb_u8.astype(np.float32)
    od = -np.log((rgb_float + 1.0) / 255.0)
    od_sum = od.sum(axis=2)

    # ============================================================
    # 3. Máscara inicial: tejido teñido o tejido pálido
    # ============================================================
    mask_sat = (
        (S > sat_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_od = (
        (od_sum > od_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_raw = cv2.bitwise_or(mask_sat, mask_od)

    # ============================================================
    # 4. Morfología
    # ============================================================
    mask_morph = mask_raw.copy()

    if open_kernel_size > 0:
        kernel_open = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (open_kernel_size, open_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size > 0:
        kernel_close = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (close_kernel_size, close_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_CLOSE, kernel_close)

    # ============================================================
    # 5. Componentes conectados
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_morph,
        connectivity=8
    )

    mask_components = np.zeros_like(mask_morph, dtype=np.uint8)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area >= min_area:
            mask_components[labels == label_id] = 255

    if np.count_nonzero(mask_components) == 0:
        raise ValueError(
            "No se detectó tejido. Prueba sat_thresh=5, od_thresh=0.012, val_thresh=255."
        )

    # ============================================================
    # 6. Extraer contorno externo y rellenarlo
    # ============================================================
    contours, _ = cv2.findContours(
        mask_components,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        raise ValueError("No se encontraron contornos de tejido.")

    # Normalmente el tejido principal es el mayor contorno
    largest_contour = max(contours, key=cv2.contourArea)

    mask_contour = np.zeros_like(mask_components, dtype=np.uint8)

    if use_convex_hull:
        hull = cv2.convexHull(largest_contour)
        cv2.drawContours(mask_contour, [hull], -1, 255, thickness=cv2.FILLED)
    else:
        cv2.drawContours(mask_contour, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # Dilatar ligeramente para no cortar borde
    mask_final = mask_contour.copy()

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    # ============================================================
    # 7. Aplicar máscara: tejido visible, resto negro
    # ============================================================
    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if debug:
        od_vis = (od_sum - od_sum.min()) / (od_sum.max() - od_sum.min() + 1e-8)

        plt.figure(figsize=(18, 10))

        plt.subplot(2, 4, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(2, 4, 2)
        plt.imshow(S, cmap="gray")
        plt.title("Saturación HSV")
        plt.axis("off")

        plt.subplot(2, 4, 3)
        plt.imshow(od_vis, cmap="gray")
        plt.title("Optical density")
        plt.axis("off")

        plt.subplot(2, 4, 4)
        plt.imshow(mask_raw, cmap="gray")
        plt.title("Máscara inicial")
        plt.axis("off")

        plt.subplot(2, 4, 5)
        plt.imshow(mask_morph, cmap="gray")
        plt.title("Después morfología")
        plt.axis("off")

        plt.subplot(2, 4, 6)
        plt.imshow(mask_components, cmap="gray")
        plt.title("Componentes filtrados")
        plt.axis("off")

        plt.subplot(2, 4, 7)
        plt.imshow(mask_final, cmap="gray")
        plt.title("Máscara final por contorno")
        plt.axis("off")

        plt.subplot(2, 4, 8)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original and show_result:
        plt.figure(figsize=(10, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")
        plt.show()

    # ============================================================
    # 9. Return limpio
    # ============================================================
    outputs = [tissue_only_rgb]

    if return_mask:
        outputs.append(mask_final)

    if return_info:
        info = {
            "chosen_level": chosen_level,
            "read_dimensions": (level_w, level_h),
            "sat_thresh": sat_thresh,
            "val_thresh": val_thresh,
            "od_thresh": od_thresh,
            "grow_pixels": grow_pixels,
            "use_convex_hull": use_convex_hull,
            "mask_area_px": int(np.count_nonzero(mask_final)),
        }
        outputs.append(info)

    if len(outputs) == 1:
        return tissue_only_rgb

    return tuple(outputs)

## Funciones del flujo H&E + HSI

In [ ]:
def read_he_preview(he_path):
    """Abre la H&E igual que en los notebooks anteriores: PIL.Image.open sobre el .mrxs."""
    he_path = Path(he_path)
    if not he_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {he_path}')

    img = Image.open(he_path)
    img_format = img.format
    rgb = np.asarray(img.convert('RGB'), dtype=np.uint8)
    info = {
        'format': img_format,
        'shape': rgb.shape,
        'mode': 'RGB',
    }
    return rgb, info


def plot_single_rgb(rgb, title, figsize=(7, 6)):
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    ax.imshow(rgb)
    ax.set_title(title)
    ax.axis('off')
    plt.show()


def resolve_he_openslide_path(he_path):
    """Devuelve una ruta H&E que OpenSlide pueda abrir, con fallback al MRXS compatible si existe."""
    he_path = Path(he_path)
    candidates = [he_path]

    compatible = he_path.parent.parent / 'H&E_python_EDU_creado' / he_path.name
    if compatible not in candidates:
        candidates.append(compatible)

    errors = []
    for candidate in candidates:
        if not candidate.exists():
            errors.append(f'{candidate} -> no existe')
            continue
        try:
            slide = openslide.OpenSlide(str(candidate))
            slide.close()
            return candidate
        except Exception as exc:
            errors.append(f'{candidate} -> {type(exc).__name__}: {exc}')

    raise RuntimeError('No se pudo abrir la H&E con OpenSlide:\n' + '\n'.join(errors))


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _find_mirax_mpp(properties, axis):
    suffix = f'MICROMETER_PER_PIXEL_{axis.upper()}'
    for key, value in properties.items():
        if key.upper().endswith(suffix):
            parsed = _safe_float(value)
            if parsed is not None:
                return parsed, key
    return None, None


def print_he_scale_info(he_path):
    """Imprime dimensiones y escala real disponible en los metadatos de la H&E."""
    openslide_path = resolve_he_openslide_path(he_path)
    slide = openslide.OpenSlide(str(openslide_path))

    try:
        props = dict(slide.properties)
        level_dimensions = list(slide.level_dimensions)
        level_downsamples = [float(v) for v in slide.level_downsamples]
        width_px, height_px = slide.dimensions

        mpp_x = _safe_float(props.get('openslide.mpp-x'))
        mpp_y = _safe_float(props.get('openslide.mpp-y'))
        if mpp_x is None:
            mpp_x, mpp_x_key = _find_mirax_mpp(props, 'x')
        else:
            mpp_x_key = 'openslide.mpp-x'
        if mpp_y is None:
            mpp_y, mpp_y_key = _find_mirax_mpp(props, 'y')
        else:
            mpp_y_key = 'openslide.mpp-y'

        print('2. Medidas H&E / escala real')
        print(f'Ruta OpenSlide usada: {openslide_path}')
        print(f'Dimensiones nivel 0: {width_px} x {height_px} px')
        print(f'Niveles disponibles: {level_dimensions}')
        print(f'Downsamples: {[round(v, 3) for v in level_downsamples]}')

        if mpp_x is not None and mpp_y is not None:
            width_mm = width_px * mpp_x / 1000.0
            height_mm = height_px * mpp_y / 1000.0
            print(f'Micras/pixel X: {mpp_x:.6f} ({mpp_x_key})')
            print(f'Micras/pixel Y: {mpp_y:.6f} ({mpp_y_key})')
            print(f'Campo nivel 0 aprox.: {width_mm:.2f} mm x {height_mm:.2f} mm')
            print(f'Escala nivel 0: {10000.0 / mpp_x:.1f} px/cm horizontal | {10000.0 / mpp_y:.1f} px/cm vertical')
        else:
            print('No se encontraron metadatos micras/pixel suficientes para calcular escala real.')

        objective = props.get('openslide.objective-power') or props.get('aperio.AppMag') or props.get('mirax.GENERAL.OBJECTIVE_MAGNIFICATION')
        if objective is not None:
            print(f'Objetivo/magnificacion: {objective}')

        return {
            'openslide_path': openslide_path,
            'dimensions_level_0': (int(width_px), int(height_px)),
            'level_dimensions': level_dimensions,
            'level_downsamples': level_downsamples,
            'mpp_x': mpp_x,
            'mpp_y': mpp_y,
        }
    finally:
        slide.close()


def bbox_corners_xyxy(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([
        [x1, y1],
        [x2, y1],
        [x2, y2],
        [x1, y2],
    ], dtype=np.float32)


def draw_dimension_line(ax, p1, p2, label, color, text_offset=(0, 0)):
    ax.annotate(
        '',
        xy=p2,
        xytext=p1,
        arrowprops=dict(arrowstyle='<->', color=color, lw=2.5, shrinkA=0, shrinkB=0),
    )
    mid = (np.asarray(p1, dtype=float) + np.asarray(p2, dtype=float)) / 2.0
    ax.text(
        mid[0] + text_offset[0],
        mid[1] + text_offset[1],
        label,
        color=color,
        fontsize=12,
        weight='bold',
        ha='center',
        va='center',
        bbox=dict(facecolor='black', alpha=0.55, edgecolor='none', pad=3),
    )


def plot_adjusted_box_with_real_measures(
    adjusted_result,
    length_cm=4.15,
    width_cm=2.85,
    title=None,
):
    """
    Dibuja la caja ajustada y asigna 4.15 cm al lado largo y 2.85 cm al lado corto.

    De momento se asume que la caja ajustada esta representada por su bbox horizontal/vertical.
    Si el lado horizontal es mayor que el vertical, 4.15 cm se asigna arriba/abajo.
    Si el vertical es mayor, 4.15 cm se asigna izquierda/derecha.
    """
    rgb = adjusted_result['pseudo_rgb']
    bbox = adjusted_result['bbox_xyxy_shrunk']
    x1, y1, x2, y2 = bbox
    box_w_px = x2 - x1 + 1
    box_h_px = y2 - y1 + 1

    horizontal_is_long = box_w_px >= box_h_px
    if horizontal_is_long:
        horizontal_cm = length_cm
        vertical_cm = width_cm
        horizontal_label = f'{length_cm:.2f} cm | lado largo'
        vertical_label = f'{width_cm:.2f} cm | lado corto'
    else:
        horizontal_cm = width_cm
        vertical_cm = length_cm
        horizontal_label = f'{width_cm:.2f} cm | lado corto'
        vertical_label = f'{length_cm:.2f} cm | lado largo'

    px_per_cm_horizontal = box_w_px / horizontal_cm
    px_per_cm_vertical = box_h_px / vertical_cm

    fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
    ax.imshow(rgb)

    overlay = adjusted_result['overlay_shrunk']
    ax.imshow(overlay, alpha=0.42)

    corners = bbox_corners_xyxy(bbox)
    closed = np.vstack([corners, corners[0]])
    ax.plot(closed[:, 0], closed[:, 1], color='yellow', linewidth=2.5)
    ax.scatter(corners[:, 0], corners[:, 1], color='yellow', s=35)

    offset = 32
    top_y = max(y1 - offset, 5)
    left_x = max(x1 - offset, 5)

    draw_dimension_line(
        ax,
        (x1, top_y),
        (x2, top_y),
        horizontal_label,
        color='lime',
        text_offset=(0, -18),
    )
    draw_dimension_line(
        ax,
        (left_x, y1),
        (left_x, y2),
        vertical_label,
        color='cyan',
        text_offset=(-48, 0),
    )

    ax.text(
        x1,
        min(y2 + 35, rgb.shape[0] - 5),
        f'Escala horizontal: {px_per_cm_horizontal:.1f} px/cm\nEscala vertical: {px_per_cm_vertical:.1f} px/cm',
        color='white',
        fontsize=11,
        ha='left',
        va='top',
        bbox=dict(facecolor='black', alpha=0.65, edgecolor='none', pad=4),
    )

    ax.set_title(title or '6. Caja azul ajustada con medidas reales')
    ax.axis('off')
    plt.show()

    return {
        'bbox_xyxy': bbox,
        'bbox_width_px': int(box_w_px),
        'bbox_height_px': int(box_h_px),
        'horizontal_cm': float(horizontal_cm),
        'vertical_cm': float(vertical_cm),
        'length_cm': float(length_cm),
        'width_cm': float(width_cm),
        'horizontal_is_long': bool(horizontal_is_long),
        'px_per_cm_horizontal': float(px_per_cm_horizontal),
        'px_per_cm_vertical': float(px_per_cm_vertical),
    }


def open_he_hsi_detect_and_measure_box(
    he_path,
    hsi_hdr_path,
    specimen_id='specimen',
    length_cm=4.15,
    width_cm=2.85,
    shrink_pixels=10,
    bbox_padding=0,
    show=True,
):
    """
    Flujo de salida:
        1. Plot H&E sola.
        2. Print de medidas/escala H&E.
        3. Segmentacion H&E usando 3_Segmentacion_H&E.ipynb.
        4. Plot HSI basica.
        5. Plot HSI con caja ajustada.
        6. Plot de asignacion de medidas reales.
    """
    he_rgb, he_info = read_he_preview(he_path)

    if show:
        plot_single_rgb(
            he_rgb,
            f'1. {specimen_id} - H&E\n{he_info["format"]} | {he_info["shape"][1]}x{he_info["shape"][0]} px',
            figsize=(6, 8),
        )

    he_scale_info = print_he_scale_info(he_path)

    he_tissue_rgb, he_tissue_mask, he_tissue_info = extract_he_tissue_contour_image(
        he_scale_info['openslide_path'],
        max_dim=1800,
        show_original=False,
        show_result=False,
        debug=False,
        return_mask=True,
        return_info=True,
    )

    if show:
        plot_single_rgb(
            he_tissue_rgb,
            f'3. {specimen_id} - H&E segmentada',
            figsize=(6, 8),
        )

    detection_result = open_hsi_and_detect_blue_box(
        hsi_hdr_path,
        specimen_id=specimen_id,
        show=False,
    )

    with io.StringIO() as _buffer, redirect_stdout(_buffer):
        adjusted_results = shrink_all_blue_box_detections(
            {specimen_id: detection_result},
            shrink_pixels=shrink_pixels,
            bbox_padding=bbox_padding,
            show=False,
        )
    adjusted_result = adjusted_results[specimen_id]

    if show:
        plot_single_rgb(
            detection_result['pseudo_rgb'],
            f'4. {specimen_id} - HSI basica\nR=651.9, G=548.4, B=449.5',
            figsize=(8, 6),
        )
        plot_single_rgb(
            adjusted_result['overlay_shrunk'],
            f'5. {specimen_id} - HSI | caja ajustada\nbbox={adjusted_result["bbox_xyxy_shrunk"]}',
            figsize=(8, 6),
        )

    measure_info = plot_adjusted_box_with_real_measures(
        adjusted_result,
        length_cm=length_cm,
        width_cm=width_cm,
        title=f'6. {specimen_id} - asignacion de medidas reales',
    )

    return {
        'specimen_id': specimen_id,
        'he_rgb': he_rgb,
        'he_info': he_info,
        'he_scale_info': he_scale_info,
        'he_tissue_rgb': he_tissue_rgb,
        'he_tissue_mask': he_tissue_mask,
        'he_tissue_info': he_tissue_info,
        'detection_result': detection_result,
        'adjusted_result': adjusted_result,
        'measure_info': measure_info,
    }


## Ejemplo de uso

La celda siguiente genera, en orden, las seis salidas pedidas: H&E, escala H&E, segmentacion H&E, HSI basica, caja ajustada y asignacion de medidas reales.

In [ ]:
# Ejemplo. Cambia EXAMPLE_ID por SB012, SB013, SB017, SB018, SB019 o SB020.
EXAMPLE_ID = 'SB013'

example_paths = SPECIMENS[EXAMPLE_ID]
example_output = open_he_hsi_detect_and_measure_box(
    he_path=example_paths['he_path'],
    hsi_hdr_path=example_paths['hsi_hdr_path'],
    specimen_id=EXAMPLE_ID,
    length_cm=4.15,
    width_cm=2.85,
    shrink_pixels=10,
    bbox_padding=0,
    show=True,
)

example_output['measure_info']



## Igualar escala real H&E vs HSI

Esta parte no cambia nada de lo anterior. Toma el resultado `example_output`, calcula la escala real de la H&E y de la HSI, y devuelve dos recortes reescalados al mismo `px/cm`.

La H&E usa `micras/pixel * downsample` del nivel que se ha segmentado. La HSI usa la caja azul ajustada, asumiendo que su lado largo mide 4.15 cm y su lado corto 2.85 cm.

In [ ]:
def mask_bbox_xyxy(mask, padding=0):
    """Obtiene bbox inclusiva (x1, y1, x2, y2) alrededor de una mascara."""
    mask_bool = np.asarray(mask) > 0
    ys, xs = np.where(mask_bool)
    if len(xs) == 0:
        raise ValueError('La mascara esta vacia; no se puede calcular el recorte.')

    h, w = mask_bool.shape[:2]
    x1 = max(int(xs.min()) - padding, 0)
    y1 = max(int(ys.min()) - padding, 0)
    x2 = min(int(xs.max()) + padding, w - 1)
    y2 = min(int(ys.max()) + padding, h - 1)
    return (x1, y1, x2, y2)


def crop_xyxy(img, bbox):
    """Recorta usando bbox inclusiva (x1, y1, x2, y2)."""
    x1, y1, x2, y2 = bbox
    return np.asarray(img)[y1:y2 + 1, x1:x2 + 1].copy()


def ensure_uint8_rgb(img):
    """Asegura una imagen RGB uint8 para visualizar/guardar."""
    arr = np.asarray(img)
    if arr.dtype == np.uint8:
        return arr
    arr = arr.astype(np.float32)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros((*arr.shape[:2], 3), dtype=np.uint8)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if arr.max() <= 1.5:
        arr = arr * 255.0
    return np.clip(arr, 0, 255).astype(np.uint8)


def resize_to_target_px_per_cm(img, src_px_per_cm_x, src_px_per_cm_y, target_px_per_cm):
    """
    Reescala una imagen para que sus ejes X/Y queden en el mismo px/cm objetivo.

    Si src_px_per_cm_x != src_px_per_cm_y, corrige esa anisotropia reescalando cada eje
    con un factor distinto. El resultado queda con pixeles isotropicos a target_px_per_cm.
    """
    img = ensure_uint8_rgb(img)
    h, w = img.shape[:2]

    scale_x = float(target_px_per_cm) / float(src_px_per_cm_x)
    scale_y = float(target_px_per_cm) / float(src_px_per_cm_y)
    new_w = max(1, int(round(w * scale_x)))
    new_h = max(1, int(round(h * scale_y)))

    interpolation = cv2.INTER_AREA if scale_x < 1.0 and scale_y < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(img, (new_w, new_h), interpolation=interpolation)
    return resized, {
        'original_shape': (int(h), int(w)),
        'scaled_shape': (int(new_h), int(new_w)),
        'scale_x': float(scale_x),
        'scale_y': float(scale_y),
        'src_px_per_cm_x': float(src_px_per_cm_x),
        'src_px_per_cm_y': float(src_px_per_cm_y),
        'target_px_per_cm': float(target_px_per_cm),
    }


def he_segmented_px_per_cm(output):
    """Calcula la escala de la H&E segmentada que devolvio extract_he_tissue_contour_image."""
    mpp_x = output['he_scale_info']['mpp_x']
    mpp_y = output['he_scale_info']['mpp_y']
    if mpp_x is None or mpp_y is None:
        raise ValueError('La H&E no tiene micras/pixel; no se puede igualar escala real.')

    chosen_level = output['he_tissue_info']['chosen_level']
    downsample = output['he_scale_info']['level_downsamples'][chosen_level]

    level_mpp_x = float(mpp_x) * float(downsample)
    level_mpp_y = float(mpp_y) * float(downsample)

    return {
        'chosen_level': int(chosen_level),
        'downsample': float(downsample),
        'mpp_x_at_level': float(level_mpp_x),
        'mpp_y_at_level': float(level_mpp_y),
        'px_per_cm_x': float(10000.0 / level_mpp_x),
        'px_per_cm_y': float(10000.0 / level_mpp_y),
    }


def hsi_box_px_per_cm(output):
    """Obtiene la escala de la HSI usando la caja azul ajustada de 4.15 x 2.85 cm."""
    measure = output['measure_info']
    return {
        'px_per_cm_x': float(measure['px_per_cm_horizontal']),
        'px_per_cm_y': float(measure['px_per_cm_vertical']),
        'horizontal_cm': float(measure['horizontal_cm']),
        'vertical_cm': float(measure['vertical_cm']),
        'bbox_xyxy': tuple(output['adjusted_result']['bbox_xyxy_shrunk']),
    }


def choose_target_px_per_cm(he_scale, hsi_scale, target='hsi'):
    """Elige la escala comun final."""
    if isinstance(target, (int, float)):
        return float(target)
    if target == 'hsi':
        return float((hsi_scale['px_per_cm_x'] + hsi_scale['px_per_cm_y']) / 2.0)
    if target == 'he':
        return float((he_scale['px_per_cm_x'] + he_scale['px_per_cm_y']) / 2.0)
    if target == 'min':
        return float(min(
            he_scale['px_per_cm_x'], he_scale['px_per_cm_y'],
            hsi_scale['px_per_cm_x'], hsi_scale['px_per_cm_y'],
        ))
    raise ValueError("target debe ser 'hsi', 'he', 'min' o un numero px/cm.")


def plot_equal_scale_pair(he_scaled, hsi_scaled, info):
    """Muestra los dos recortes ya reescalados. Las matrices resultantes comparten px/cm."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

    axes[0].imshow(he_scaled)
    axes[0].set_title(
        f"H&E segmentada | misma escala\n"
        f"{he_scaled.shape[1]}x{he_scaled.shape[0]} px | {info['target_px_per_cm']:.1f} px/cm"
    )
    axes[0].axis('off')

    axes[1].imshow(hsi_scaled)
    axes[1].set_title(
        f"HSI caja/especimen | misma escala\n"
        f"{hsi_scaled.shape[1]}x{hsi_scaled.shape[0]} px | {info['target_px_per_cm']:.1f} px/cm"
    )
    axes[1].axis('off')

    plt.show()


def equalize_he_hsi_real_scale(
    output,
    target_px_per_cm='hsi',
    he_padding_px=20,
    hsi_padding_px=0,
    show=True,
):
    """
    Genera dos imagenes a la misma escala real:
        - H&E: recorte del tejido segmentado.
        - HSI: recorte calibrado por la caja azul ajustada.

    Importante: la escala fisica queda en las matrices devueltas. Es decir, si ambas
    imagenes se guardan o se comparan pixel a pixel, comparten el mismo px/cm.
    """
    he_scale = he_segmented_px_per_cm(output)
    hsi_scale = hsi_box_px_per_cm(output)
    target = choose_target_px_per_cm(he_scale, hsi_scale, target=target_px_per_cm)

    he_bbox = mask_bbox_xyxy(output['he_tissue_mask'], padding=he_padding_px)
    he_crop = crop_xyxy(output['he_tissue_rgb'], he_bbox)

    hsi_bbox = hsi_scale['bbox_xyxy']
    if hsi_padding_px:
        hsi_img = output['adjusted_result']['pseudo_rgb']
        x1, y1, x2, y2 = hsi_bbox
        h, w = hsi_img.shape[:2]
        hsi_bbox = (
            max(0, x1 - hsi_padding_px),
            max(0, y1 - hsi_padding_px),
            min(w - 1, x2 + hsi_padding_px),
            min(h - 1, y2 + hsi_padding_px),
        )
    hsi_crop = crop_xyxy(output['adjusted_result']['pseudo_rgb'], hsi_bbox)

    he_scaled, he_resize_info = resize_to_target_px_per_cm(
        he_crop,
        src_px_per_cm_x=he_scale['px_per_cm_x'],
        src_px_per_cm_y=he_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )
    hsi_scaled, hsi_resize_info = resize_to_target_px_per_cm(
        hsi_crop,
        src_px_per_cm_x=hsi_scale['px_per_cm_x'],
        src_px_per_cm_y=hsi_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )

    info = {
        'target_px_per_cm': float(target),
        'he_scale': he_scale,
        'hsi_scale': hsi_scale,
        'he_bbox_xyxy': he_bbox,
        'hsi_bbox_xyxy': hsi_bbox,
        'he_resize_info': he_resize_info,
        'hsi_resize_info': hsi_resize_info,
        'he_real_width_cm': float(he_crop.shape[1] / he_scale['px_per_cm_x']),
        'he_real_height_cm': float(he_crop.shape[0] / he_scale['px_per_cm_y']),
        'hsi_real_width_cm': float(hsi_crop.shape[1] / hsi_scale['px_per_cm_x']),
        'hsi_real_height_cm': float(hsi_crop.shape[0] / hsi_scale['px_per_cm_y']),
    }

    print('Escala comun final')
    print(f"  target: {target:.2f} px/cm")
    print(f"  H&E original: {he_scale['px_per_cm_x']:.2f} px/cm X | {he_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  HSI original: {hsi_scale['px_per_cm_x']:.2f} px/cm X | {hsi_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  H&E recorte real aprox.: {info['he_real_width_cm']:.2f} x {info['he_real_height_cm']:.2f} cm")
    print(f"  HSI recorte real aprox.: {info['hsi_real_width_cm']:.2f} x {info['hsi_real_height_cm']:.2f} cm")

    if show:
        plot_equal_scale_pair(he_scaled, hsi_scaled, info)

    return {
        'he_scaled_rgb': he_scaled,
        'hsi_scaled_rgb': hsi_scaled,
        'info': info,
    }


## Ejemplo: dos im?genes a la misma escala real

Este ejemplo usa el `example_output` ya calculado arriba. Por defecto toma como escala final la media de la HSI calibrada por la caja azul (`target_px_per_cm='hsi'`).

In [ ]:
same_scale_output = equalize_he_hsi_real_scale(
    example_output,
    target_px_per_cm='hsi',
    he_padding_px=20,
    hsi_padding_px=0,
    show=True,
)

same_scale_output['info']


## Comprobaciones de escala real

Estas comprobaciones sirven para distinguir dos cosas:

1. Que las matrices resultantes esten realmente en el mismo `px/cm`.
2. Que el plot no este agrandando visualmente una imagen pequena por meterla en un subplot del mismo tama?o que la otra.

La visualizacion de lienzo unico pega ambas imagenes en una sola matriz, asi que conserva las proporciones en pixeles entre H&E y HSI.

In [ ]:
def print_equal_scale_diagnostics(same_scale_output):
    """Imprime comprobaciones numericas de escala para H&E y HSI ya reescaladas."""
    he = same_scale_output['he_scaled_rgb']
    hsi = same_scale_output['hsi_scaled_rgb']
    info = same_scale_output['info']
    target = info['target_px_per_cm']

    he_h, he_w = he.shape[:2]
    hsi_h, hsi_w = hsi.shape[:2]

    diagnostics = {
        'target_px_per_cm': float(target),
        'he_shape_px': (int(he_h), int(he_w)),
        'hsi_shape_px': (int(hsi_h), int(hsi_w)),
        'he_size_cm_from_pixels': (float(he_w / target), float(he_h / target)),
        'hsi_size_cm_from_pixels': (float(hsi_w / target), float(hsi_h / target)),
        'he_size_cm_before_resize': (info['he_real_width_cm'], info['he_real_height_cm']),
        'hsi_size_cm_before_resize': (info['hsi_real_width_cm'], info['hsi_real_height_cm']),
        'one_cm_bar_px': float(target),
    }

    print('Comprobacion 1: ambas imagenes finales usan el mismo px/cm')
    print(f"  Escala comun: {target:.2f} px/cm")
    print(f"  Barra de 1 cm debe medir: {target:.2f} px en ambas")
    print()
    print('Comprobacion 2: tama?o fisico calculado desde las matrices finales')
    print(f"  H&E final: {he_w}x{he_h} px -> {he_w / target:.2f} x {he_h / target:.2f} cm")
    print(f"  HSI final: {hsi_w}x{hsi_h} px -> {hsi_w / target:.2f} x {hsi_h / target:.2f} cm")
    print()
    print('Comprobacion 3: tama?o fisico antes/despues del resize')
    print(f"  H&E antes: {info['he_real_width_cm']:.2f} x {info['he_real_height_cm']:.2f} cm")
    print(f"  HSI antes: {info['hsi_real_width_cm']:.2f} x {info['hsi_real_height_cm']:.2f} cm")
    print()
    print('Interpretacion rapida')
    if he_w < hsi_w and he_h < hsi_h:
        print('  Numericamente la H&E es menor que la HSI en el lienzo final; si se ve mas grande, es efecto del subplot.')
    else:
        print('  Hay que revisar si el recorte H&E o la caja HSI estan representando la misma region anatomica.')

    return diagnostics


def add_scale_bar_to_rgb(img, px_per_cm, bar_cm=1.0, label=None, color=(255, 255, 0)):
    """A?ade una barra de escala dentro de una copia RGB."""
    out = ensure_uint8_rgb(img).copy()
    h, w = out.shape[:2]
    bar_px = max(1, int(round(px_per_cm * bar_cm)))
    bar_px = min(bar_px, max(1, w - 40))

    x1 = 20
    y1 = max(20, h - 30)
    x2 = x1 + bar_px
    cv2.line(out, (x1, y1), (x2, y1), color, thickness=4)
    cv2.line(out, (x1, y1 - 8), (x1, y1 + 8), color, thickness=3)
    cv2.line(out, (x2, y1 - 8), (x2, y1 + 8), color, thickness=3)

    text = label or f'{bar_cm:g} cm'
    cv2.putText(out, text, (x1, max(15, y1 - 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    return out


def plot_same_scale_on_one_canvas(same_scale_output, gap_px=80, background=255):
    """
    Pega H&E y HSI en un unico lienzo, respetando tama?os en pixeles.

    Esto evita el efecto visual de los subplots, donde cada axes se dibuja con tama?o parecido
    aunque las matrices tengan anchuras/alturas distintas.
    """
    he = same_scale_output['he_scaled_rgb']
    hsi = same_scale_output['hsi_scaled_rgb']
    target = same_scale_output['info']['target_px_per_cm']

    he_bar = add_scale_bar_to_rgb(he, target, bar_cm=1.0, label='1 cm')
    hsi_bar = add_scale_bar_to_rgb(hsi, target, bar_cm=1.0, label='1 cm')

    he_h, he_w = he_bar.shape[:2]
    hsi_h, hsi_w = hsi_bar.shape[:2]
    label_h = 42
    canvas_h = label_h + max(he_h, hsi_h)
    canvas_w = he_w + gap_px + hsi_w
    canvas = np.full((canvas_h, canvas_w, 3), background, dtype=np.uint8)

    he_y = label_h + (max(he_h, hsi_h) - he_h) // 2
    hsi_y = label_h + (max(he_h, hsi_h) - hsi_h) // 2
    he_x = 0
    hsi_x = he_w + gap_px

    canvas[he_y:he_y + he_h, he_x:he_x + he_w] = he_bar
    canvas[hsi_y:hsi_y + hsi_h, hsi_x:hsi_x + hsi_w] = hsi_bar

    cv2.putText(canvas, f'H&E {he_w}x{he_h}px', (he_x + 5, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(canvas, f'HSI {hsi_w}x{hsi_h}px', (hsi_x + 5, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 2, cv2.LINE_AA)

    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    ax.imshow(canvas)
    ax.set_title(f'Comprobacion en lienzo unico | misma escala: {target:.1f} px/cm')
    ax.axis('off')
    plt.show()

    return canvas


In [ ]:
scale_diagnostics = print_equal_scale_diagnostics(same_scale_output)
same_scale_canvas = plot_same_scale_on_one_canvas(same_scale_output)

scale_diagnostics


## Igualar escala real usando HSI segmentada

Esta seccion a?ade la segmentacion HSI del notebook `2_Segmentacion_HSI.ipynb`. La escala real no se calcula desde la mascara segmentada, sino desde la caja azul ajustada: la mascara solo decide que pixeles del especimen HSI se conservan.

Asi mantenemos la calibracion fisica: cada pixel HSI segmentado sigue viviendo en el sistema de coordenadas calibrado por la caja de `4.15 x 2.85 cm`.

### Funcion de segmentacion HSI reutilizada

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False
):
    """
    Carga una imagen hiperespectral ENVI desde una ruta .hdr y devuelve una pseudo-RGB
    donde solo se ve el espécimen; el resto queda negro.

    Parámetros
    ----------
    hdr_path : str
        Ruta al archivo .hdr de la imagen hiperespectral.

    r_nm, g_nm, b_nm : float
        Longitudes de onda usadas para construir la pseudo-RGB.

    grow_pixels : int
        Número de píxeles para agrandar ligeramente el contorno del espécimen.
        Útil para evitar cortar tejido.

    show_original : bool
        Si True, plotea la pseudo-RGB original.

    show_result : bool
        Si True, plotea la imagen con solo el espécimen.

    return_mask : bool
        Si True, devuelve también la máscara binaria del espécimen.

    Returns
    -------
    specimen_only_rgb : np.ndarray
        Imagen RGB uint8 con solo el espécimen visible y el resto negro.

    Opcionalmente:
    mask_specimen : np.ndarray
        Máscara uint8 del espécimen, con valores 0 y 255.
    """

    # ============================================================
    # 1. Cargar cubo HSI
    # ============================================================
    img = open_image(hdr_path)
    import warnings
    try:
        from spectral.io.spyfile import NaNValueWarning
        ignored_warnings = (NaNValueWarning,)
    except Exception:
        ignored_warnings = (Warning,)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ignored_warnings)
        cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array(
        [float(w) for w in img.metadata["wavelength"]],
        dtype=np.float32
    )

    # ============================================================
    # 2. Funciones internas
    # ============================================================
    def find_nearest_band(wavelengths, target_nm):
        wavelengths_arr = np.asarray(wavelengths, dtype=float)
        idx = np.argmin(np.abs(wavelengths_arr - target_nm))
        return idx

    def robust_normalize(channel, p_low=2, p_high=98):
        ch = channel.astype(np.float32)

        # Convertimos inf/-inf en NaN para tratarlos igual.
        ch = np.where(np.isfinite(ch), ch, np.nan)

        # Si toda la banda es invalida, devolvemos negro.
        if np.all(np.isnan(ch)):
            return np.zeros_like(ch, dtype=np.float32)

        # Percentiles ignorando NaN.
        lo = np.nanpercentile(ch, p_low)
        hi = np.nanpercentile(ch, p_high)

        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            return np.zeros_like(ch, dtype=np.float32)

        ch = (ch - lo) / (hi - lo)
        ch = np.clip(ch, 0, 1)

        # Limpieza final para que nunca llegue NaN al astype(uint8).
        ch = np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0)

        return ch.astype(np.float32)
    # ============================================================
    # 3. Crear pseudo-RGB
    # ============================================================
    r_idx = find_nearest_band(wavelengths, r_nm)
    g_idx = find_nearest_band(wavelengths, g_nm)
    b_idx = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    pseudo_rgb = np.stack([r, g, b], axis=-1)

    pseudo_rgb = np.nan_to_num(
        pseudo_rgb,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)

    # ============================================================
    # 4. Detectar caja azul
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    R = rgb_u8[:, :, 0].astype(np.int16)
    G = rgb_u8[:, :, 1].astype(np.int16)
    B = rgb_u8[:, :, 2].astype(np.int16)

    V = hsv[:, :, 2]

    lower_blue = np.array([85, 10, 40], dtype=np.uint8)
    upper_blue = np.array([125, 180, 255], dtype=np.uint8)

    mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)

    mask_dominance = (
        ((B - R) > 8) &
        ((G - R) > 3) &
        (V > 50)
    )

    mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
    mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

    if np.count_nonzero(mask_blue) == 0:
        raise ValueError("No se detectó la caja azul. Revisa los umbrales HSV.")

    # ============================================================
    # 5. Quedarse con el componente azul más grande
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_blue,
        connectivity=8
    )

    if num_labels <= 1:
        raise ValueError("No se encontró ningún componente azul válido.")

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + np.argmax(areas)

    mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

    # ============================================================
    # 6. Contorno interno de la caja azul = espécimen
    # ============================================================
    contours, hierarchy = cv2.findContours(
        mask_box_blue,
        cv2.RETR_CCOMP,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if hierarchy is None:
        raise ValueError("No se encontraron contornos en la máscara de la caja azul.")

    hierarchy = hierarchy[0]

    inner_contours = [
        cnt for cnt, h in zip(contours, hierarchy)
        if h[3] != -1
    ]

    mask_specimen = np.zeros_like(mask_box_blue, dtype=np.uint8)

    if len(inner_contours) > 0:
        specimen_contour = max(inner_contours, key=cv2.contourArea)
        cv2.drawContours(
            mask_specimen,
            [specimen_contour],
            -1,
            255,
            thickness=cv2.FILLED
        )
    else:
        # Fallback para casos como SB017: la caja azul no siempre forma un hueco interno cerrado.
        # En ese caso segmentamos el tejido dentro del bounding box de la caja, descartando azul,
        # negro de fondo y zonas blancas/poco saturadas.
        x = stats[largest_label, cv2.CC_STAT_LEFT]
        y = stats[largest_label, cv2.CC_STAT_TOP]
        w = stats[largest_label, cv2.CC_STAT_WIDTH]
        h = stats[largest_label, cv2.CC_STAT_HEIGHT]

        pad = 5
        H_img, W_img = rgb_u8.shape[:2]
        x1 = max(x - pad, 0)
        y1 = max(y - pad, 0)
        x2 = min(x + w + pad, W_img)
        y2 = min(y + h + pad, H_img)

        crop_rgb = rgb_u8[y1:y2, x1:x2]
        crop_blue = mask_box_blue[y1:y2, x1:x2] > 0
        crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)

        Hc = crop_hsv[:, :, 0]
        Sc = crop_hsv[:, :, 1]
        Vc = crop_hsv[:, :, 2]
        Rc = crop_rgb[:, :, 0].astype(np.int16)
        Gc = crop_rgb[:, :, 1].astype(np.int16)
        Bc = crop_rgb[:, :, 2].astype(np.int16)

        tissue_color = (
            ((Rc > Bc + 8) | (Gc > Bc + 8) | (Rc > Gc + 8)) &
            (Sc > 25) &
            (Vc > 25) &
            (Vc < 245)
        )
        not_blue_box = ~crop_blue
        not_white_background = ~((Sc < 35) & (Vc > 170))
        not_black_background = Vc > 25

        mask_crop = (tissue_color & not_blue_box & not_white_background & not_black_background).astype(np.uint8) * 255

        kernel_open_tissue = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        kernel_close_tissue = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
        mask_crop = cv2.morphologyEx(mask_crop, cv2.MORPH_OPEN, kernel_open_tissue)
        mask_crop = cv2.morphologyEx(mask_crop, cv2.MORPH_CLOSE, kernel_close_tissue)

        num_tissue_labels, tissue_labels, tissue_stats, _ = cv2.connectedComponentsWithStats(mask_crop, connectivity=8)
        if num_tissue_labels <= 1:
            raise ValueError("No se encontró hueco interno y tampoco se pudo segmentar el tejido dentro de la caja azul.")

        tissue_areas = tissue_stats[1:, cv2.CC_STAT_AREA]
        tissue_label = 1 + np.argmax(tissue_areas)
        mask_crop_clean = (tissue_labels == tissue_label).astype(np.uint8) * 255

        contours_tissue, _ = cv2.findContours(mask_crop_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours_tissue) == 0:
            raise ValueError("No se encontró contorno del tejido en el fallback de segmentación.")

        tissue_contour = max(contours_tissue, key=cv2.contourArea)
        mask_crop_clean = np.zeros_like(mask_crop_clean, dtype=np.uint8)
        cv2.drawContours(mask_crop_clean, [tissue_contour], -1, 255, thickness=cv2.FILLED)

        mask_specimen[y1:y2, x1:x2] = mask_crop_clean

    # Suavizar pequeñas irregularidades
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_specimen = cv2.morphologyEx(
        mask_specimen,
        cv2.MORPH_CLOSE,
        kernel_smooth
    )

    # Agrandar ligeramente el contorno
    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )

        mask_specimen = cv2.dilate(
            mask_specimen,
            kernel_grow,
            iterations=1
        )

    # ============================================================
    # 7. Aplicar máscara: espécimen visible, resto negro
    # ============================================================
    specimen_only_rgb = rgb_u8.copy()
    specimen_only_rgb[mask_specimen == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if show_original and show_result:
        plt.figure(figsize=(12, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")
        plt.show()

    if return_mask:
        return specimen_only_rgb, mask_specimen

    return specimen_only_rgb


### Comparacion H&E segmentada vs HSI segmentada a la misma escala

In [ ]:
def equalize_he_hsi_segmented_real_scale(
    output,
    hsi_hdr_path,
    target_px_per_cm='hsi',
    he_padding_px=20,
    hsi_padding_px=10,
    hsi_grow_pixels=5,
    show=True,
):
    """
    Igual que equalize_he_hsi_real_scale, pero usando la HSI segmentada con la funcion
    de 2_Segmentacion_HSI.ipynb.

    La escala HSI se conserva desde la caja azul ajustada, no desde el bbox del especimen.
    """
    hsi_specimen_rgb, hsi_specimen_mask = extract_specimen_only_from_hsi_path(
        str(hsi_hdr_path),
        grow_pixels=hsi_grow_pixels,
        show_original=False,
        show_result=False,
        return_mask=True,
    )

    he_scale = he_segmented_px_per_cm(output)
    hsi_scale = hsi_box_px_per_cm(output)
    target = choose_target_px_per_cm(he_scale, hsi_scale, target=target_px_per_cm)

    he_bbox = mask_bbox_xyxy(output['he_tissue_mask'], padding=he_padding_px)
    he_crop = crop_xyxy(output['he_tissue_rgb'], he_bbox)

    hsi_specimen_bbox = mask_bbox_xyxy(hsi_specimen_mask, padding=hsi_padding_px)
    hsi_specimen_crop = crop_xyxy(hsi_specimen_rgb, hsi_specimen_bbox)

    he_scaled, he_resize_info = resize_to_target_px_per_cm(
        he_crop,
        src_px_per_cm_x=he_scale['px_per_cm_x'],
        src_px_per_cm_y=he_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )
    hsi_segmented_scaled, hsi_resize_info = resize_to_target_px_per_cm(
        hsi_specimen_crop,
        src_px_per_cm_x=hsi_scale['px_per_cm_x'],
        src_px_per_cm_y=hsi_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )

    info = {
        'target_px_per_cm': float(target),
        'he_scale': he_scale,
        'hsi_scale': hsi_scale,
        'he_bbox_xyxy': he_bbox,
        'hsi_specimen_bbox_xyxy': hsi_specimen_bbox,
        'he_resize_info': he_resize_info,
        'hsi_resize_info': hsi_resize_info,
        'he_real_width_cm': float(he_crop.shape[1] / he_scale['px_per_cm_x']),
        'he_real_height_cm': float(he_crop.shape[0] / he_scale['px_per_cm_y']),
        'hsi_segmented_real_width_cm': float(hsi_specimen_crop.shape[1] / hsi_scale['px_per_cm_x']),
        'hsi_segmented_real_height_cm': float(hsi_specimen_crop.shape[0] / hsi_scale['px_per_cm_y']),
        'hsi_mask_area_px': int(np.count_nonzero(hsi_specimen_mask)),
    }

    print('Escala comun final con HSI segmentada')
    print(f"  target: {target:.2f} px/cm")
    print(f"  H&E original: {he_scale['px_per_cm_x']:.2f} px/cm X | {he_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  HSI original: {hsi_scale['px_per_cm_x']:.2f} px/cm X | {hsi_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  H&E recorte real aprox.: {info['he_real_width_cm']:.2f} x {info['he_real_height_cm']:.2f} cm")
    print(f"  HSI segmentada real aprox.: {info['hsi_segmented_real_width_cm']:.2f} x {info['hsi_segmented_real_height_cm']:.2f} cm")
    print(f"  HSI mascara area: {info['hsi_mask_area_px']} px")

    result = {
        'he_scaled_rgb': he_scaled,
        'hsi_scaled_rgb': hsi_segmented_scaled,
        'hsi_specimen_rgb': hsi_specimen_rgb,
        'hsi_specimen_mask': hsi_specimen_mask,
        'info': info,
    }

    if show:
        plot_same_scale_on_one_canvas(result)

    return result


In [ ]:
same_scale_segmented_hsi_output = equalize_he_hsi_segmented_real_scale(
    example_output,
    hsi_hdr_path=example_paths['hsi_hdr_path'],
    target_px_per_cm='hsi',
    he_padding_px=20,
    hsi_padding_px=10,
    hsi_grow_pixels=5,
    show=True,
)

same_scale_segmented_hsi_output['info']
